# Baseline Model with k-fold Cross Validation

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [31]:
# Import necessary libraries

import random

import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from datasets import concatenate_datasets, load_dataset


In [32]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")  # NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")   # Apple Silicon (M1/M2/M3)
else:
    DEVICE = torch.device("cpu")   # Fallback

print(f"Using Device: {DEVICE}")

Using Device: mps


## Model Choice

[Explain why you've chosen a particular model as the baseline. This could be a simple statistical model or a basic machine learning model. Justify your choice.]


We chose a convolutional neural network (CNN) as the baseline model for our aesthetic emotions map project. CNNs are well-suited for image classification tasks due to their ability to capture spatial hierarchies in images. They can learn to recognize patterns and features in the images that are relevant for predicting the associated emotions. Additionally, CNNs have been widely used and have shown strong performance in various image-related tasks, making them a reasonable starting point for our project.

## Feature Selection

[Indicate which features from the dataset you will be using for the baseline model, and justify your selection.]

For the baseline model, we will be using the raw pixel values of the images as features. This is a common approach for image classification tasks, as it allows the model to learn directly from the visual data without any manual feature engineering. By using the raw pixel values, we can leverage the CNN's ability to automatically extract relevant features from the images during training.

In [33]:
# Loading the dataset using Hugging Face's datasets library
dataset = load_dataset("bjoern-doege/aesthetic-emotions-map")

Resolving data files:   0%|          | 0/4721 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1025 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1089 [00:00<?, ?it/s]

In [34]:
NORMALIZE_MEAN=[0.5111283659934998, 0.48830345273017883, 0.46479079127311707]
NORMALIZE_STD=[0.3433663249015808, 0.3207928538322449, 0.32255250215530396]

In [35]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

VAL_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

In [36]:
# Converting labels to ids
# PyTorch's ImageFolder would do this automatically, but we are using Hugging Face's datasets library
label_names = sorted(dataset["train"].unique("label"))
label_to_id = {label: i for i, label in enumerate(label_names)}

In [37]:
# This function will be applied to each example in the dataset to preprocess the images and labels
# Again, this is necessary because we are using Hugging Face's datasets library instead of PyTorch's ImageFolder
def preprocess_train(examples):
    return {
        "image": [
            TRAIN_TRANSFORM(image.convert("RGB"))
            for image in examples["image"]
        ],
        "label": torch.tensor(
            [label_to_id[label] for label in examples["label"]],
            dtype=torch.long,
        ),
    }

def preprocess_val(examples):
    return {
        "image": [
            VAL_TRANSFORM(image.convert("RGB"))
            for image in examples["image"]
        ],
        "label": torch.tensor(
            [label_to_id[label] for label in examples["label"]],
            dtype=torch.long,
        ),
    }

In [38]:
# Use train + validation for cross-validation and keep test untouched for final evaluation.
trainval_dataset = concatenate_datasets([
    dataset["train"],
    dataset["validation"],
])

test_dataset = dataset["test"]

BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
N_SPLITS = 5
RANDOM_STATE = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(RANDOM_STATE)


In [39]:
X = np.arange(len(trainval_dataset))
y = [label_to_id[label] for label in trainval_dataset["label"]]
groups = trainval_dataset["style"]

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

style_overlap = set(trainval_dataset["style"]) & set(test_dataset["style"])
print(f"Train+validation samples: {len(trainval_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Overlapping styles between train+validation and test: {len(style_overlap)}")


Train+validation samples: 5744
Test samples: 1088
Overlapping styles between train+validation and test: 0


## Implementation

[Implement your baseline model here.]

The baseline CNN architecture consists of the following layers assuming an input image size of 224x224 pixels:
1. **Convolutional Layer 1**: 32 filters, kernel size of 3x3, ReLU activation
2. **Max Pooling Layer 1**: Pool size of 2x2
3. **Convolutional Layer 2**: 64 filters, kernel size of 3x3, ReLU activation
4. **Max Pooling Layer 2**: Pool size of 2x2
5. **Convolutional Layer 3**: 128 filters, kernel size of 3x3, ReLU activation
6. **Max Pooling Layer 3**: Pool size of 2x2
7. **Flatten Layer**: Flattens the output from the previous layer
8. **Fully Connected Layer 1**: 256 neurons, ReLU activation
9. **Dropout Layer**: Dropout rate of 0.5 to reduce overfitting
10. **Output Layer**: Number of neurons equal to the number of emotion classes

In [40]:
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

# Print the last layer of the classifier of the loaded model
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [41]:
def create_optimizer(model):
    return optim.Adam(model.parameters(), lr=LEARNING_RATE)


def compute_class_weights(ds):
    labels = [label_to_id[label] for label in ds["label"]]
    class_counts = np.bincount(labels, minlength=len(label_names))
    class_counts = np.maximum(class_counts, 1)
    weights = len(labels) / (len(label_names) * class_counts)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def create_loss_fn(ds):
    class_weights = compute_class_weights(ds)
    return nn.CrossEntropyLoss(weight=class_weights)


def create_dataloader(ds, batch_size=BATCH_SIZE, train=False, shuffle=False, seed=RANDOM_STATE):
    if train:
        transform = preprocess_train
    else:
        transform = preprocess_val

    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(seed)

    return DataLoader(
        ds.with_transform(transform),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=generator,
    )


In [42]:
def update_model_last_layer(model, num_classes):
    """
    Freezes the feature layers of a pre-trained model and replaces its final
    classification layer with a new one adapted to the specified number of classes.

    Args:
        model (torch.nn.Module): The pre-trained model to be modified.
        num_classes (int): The number of output classes for the new classification layer.

    Returns:
        torch.nn.Module: The modified model with frozen feature layers and a new
                         classification layer.
    """


    for name, param in model.named_parameters():
        if not name.startswith("fc."):
            param.requires_grad = False

    last_classifier_layer = model.fc 
    
    num_features = last_classifier_layer.in_features
    
    new_classifier = nn.Linear(in_features=num_features, out_features=num_classes)
    
    # Replace the original last classification layer with the newly created layer
    model.fc = new_classifier

    return model

In [43]:
def create_model():
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)
    model = update_model_last_layer(model, num_classes=8)
    return model.to(DEVICE)

In [44]:
# Modify the last layer of the MobileNetV3-Large model
test_model = create_model()
# test_model = update_model_last_layer(test_model, num_classes=8)

# Print the last layer of the classifier of the modified model
print(test_model.fc)

Linear(in_features=2048, out_features=8, bias=True)


In [45]:
def train_one_epoch(model, train_loader, optimizer, loss_fn, device):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    return compute_metrics(y_true, y_pred, running_loss)


def evaluate(model, data_loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in data_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            running_loss += loss.item() * images.size(0)
            predicted = outputs.argmax(dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    return compute_metrics(y_true, y_pred, running_loss)


def compute_metrics(y_true, y_pred, total_loss):
    return {
        "loss": total_loss / len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }


In [ ]:
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups), start=1):
    print(f"\nFold {fold}/{N_SPLITS}")
    fold_seed = RANDOM_STATE + fold
    seed_everything(fold_seed)

    fold_train_dataset = trainval_dataset.select(train_idx.tolist())
    fold_val_dataset = trainval_dataset.select(val_idx.tolist())

    train_loader = create_dataloader(fold_train_dataset, train=True, shuffle=True, seed=fold_seed)
    val_loader = create_dataloader(fold_val_dataset, train=False, shuffle=False)

    model = create_model()
    optimizer = create_optimizer(model)
    loss_fn = create_loss_fn(fold_train_dataset)

    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_metrics = evaluate(model, val_loader, loss_fn, DEVICE)

        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
            f"train_loss={train_metrics['loss']:.4f}, "
            f"train_acc={train_metrics['accuracy']:.4f}, "
            f"train_macro_f1={train_metrics['f1_macro']:.4f} | "
            f"train_weighted_f1={train_metrics['f1_weighted']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f}, "
            f"val_acc={val_metrics['accuracy']:.4f}, "
            f"val_macro_f1={val_metrics['f1_macro']:.4f}, "
            f"val_weighted_f1={val_metrics['f1_weighted']:.4f}"
        )

    fold_metrics.append({
        "fold": fold,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1_weighted": val_metrics["f1_weighted"],
        "val_f1_macro": val_metrics["f1_macro"],
    })

val_macro_f1_scores = [metrics["val_f1_macro"] for metrics in fold_metrics]
val_accuracy_scores = [metrics["val_accuracy"] for metrics in fold_metrics]
val_f1_weighted_scores = [metrics["val_f1_weighted"] for metrics in fold_metrics]

print("\nCross-validation summary")
print(f"Accuracy: {np.mean(val_accuracy_scores):.4f} +/- {np.std(val_accuracy_scores):.4f}")
print(f"Macro F1: {np.mean(val_macro_f1_scores):.4f} +/- {np.std(val_macro_f1_scores):.4f}")
print(f"Weighted F1: {np.mean(val_f1_weighted_scores):.4f} +/- {np.std(val_f1_weighted_scores):.4f}")


Fold 1/5
Epoch 1/10 | train_loss=1.6525, train_acc=0.4288, train_macro_f1=0.4146 | val_loss=1.6195, val_acc=0.4141, val_macro_f1=0.3903
Epoch 2/10 | train_loss=1.2189, train_acc=0.6006, train_macro_f1=0.5892 | val_loss=1.5011, val_acc=0.4488, val_macro_f1=0.4194
Epoch 3/10 | train_loss=1.0543, train_acc=0.6533, train_macro_f1=0.6448 | val_loss=1.4740, val_acc=0.4948, val_macro_f1=0.4484


KeyboardInterrupt: 

## Evaluation

[Clearly state what metrics you will use to evaluate the model's performance. These metrics will serve as a starting point for evaluating more complex models later on.]

We are using the following metrics to evaluate the performance of our baseline model:
1. **Accuracy**: The proportion of correctly classified instances among the total instances.
2. **Precision**: The proportion of true positive predictions among all positive predictions.
3. **Recall**: The proportion of true positive predictions among all actual positives.
4. **F1 Score**: The harmonic mean of precision and recall, providing a balance between the two metrics.
5. **Confusion Matrix**: A table that describes the performance of the classification model by showing the true positives, true negatives, false positives, and false negatives for each class.
6. **Classification Report**: A comprehensive report that includes precision, recall, F1 score, and support for each class, providing a detailed overview of the model's performance across all classes.

In [ ]:
# Train one final model on all train+validation data, then evaluate once on the untouched test set.
final_seed = RANDOM_STATE + N_SPLITS + 1
seed_everything(final_seed)
final_model = create_model()
final_optimizer = create_optimizer(final_model)
final_loss_fn = create_loss_fn(trainval_dataset)

trainval_loader = create_dataloader(trainval_dataset, shuffle=True, seed=final_seed)
test_loader = create_dataloader(test_dataset, shuffle=False)

for epoch in range(NUM_EPOCHS):
    train_metrics = train_one_epoch(final_model, trainval_loader, final_optimizer, final_loss_fn, DEVICE)
    print(
        f"Final model epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f}, "
        f"train_acc={train_metrics['accuracy']:.4f}, "
        f"train_macro_f1={train_metrics['f1_macro']:.4f}"
    )

test_metrics = evaluate(final_model, test_loader, final_loss_fn, DEVICE)

print("\nFinal test metrics")
print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Weighted precision: {test_metrics['precision_weighted']:.4f}")
print(f"Weighted recall: {test_metrics['recall_weighted']:.4f}")
print(f"Weighted F1: {test_metrics['f1_weighted']:.4f}")
print(f"Macro F1: {test_metrics['f1_macro']:.4f}")

print("\nClassification report:")
print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0,
))

print("\nConfusion matrix:")
print(confusion_matrix(test_metrics["y_true"], test_metrics["y_pred"]))


Final model epoch 1/10 | train_loss=1.4486, train_acc=0.4445, train_macro_f1=0.4342
Final model epoch 2/10 | train_loss=1.0198, train_acc=0.6149, train_macro_f1=0.6092
Final model epoch 3/10 | train_loss=0.8259, train_acc=0.6903, train_macro_f1=0.6873
Final model epoch 4/10 | train_loss=0.6571, train_acc=0.7604, train_macro_f1=0.7589
Final model epoch 5/10 | train_loss=0.5413, train_acc=0.8024, train_macro_f1=0.8024
Final model epoch 6/10 | train_loss=0.4375, train_acc=0.8402, train_macro_f1=0.8440
Final model epoch 7/10 | train_loss=0.3581, train_acc=0.8701, train_macro_f1=0.8740
Final model epoch 8/10 | train_loss=0.3096, train_acc=0.8877, train_macro_f1=0.8912
Final model epoch 9/10 | train_loss=0.2545, train_acc=0.9088, train_macro_f1=0.9109
Final model epoch 10/10 | train_loss=0.2093, train_acc=0.9295, train_macro_f1=0.9342

Final test metrics
Test loss: 1.9255
Accuracy: 0.4908
Weighted precision: 0.4971
Weighted recall: 0.4908
Weighted F1: 0.4920
Macro F1: 0.4507

Classification 

In [ ]:
torch.save(final_model.state_dict(), "model_seed_42_w_k-fold_transfer_learning.pth")
